# Block-Accumulator vs Per-Param vs Adam-QAT — GPU Comparison

Compares three MXFP4 training arms on the same char-LM:
1. **MXFP4 + block-acc** (4.50 bits/param, 11-bit block-shared accumulator)
2. **MXFP4 + per-param 9-state** (8.25 bits/param, per-weight integrator)
3. **MXFP4 + per-param 3-state** (6.25 bits/param, per-weight integrator)
4. **Adam-QAT** (MXFP4 forward + fp32 master + AdamW, ~100 bits/param)
5. **SR-only** (4.25 bits/param, no accumulator — the baseline to beat)

See `docs/BLOCK_ACCUMULATOR.md` for the design.

**Run on Colab**: select a GPU runtime (T4 or A100), then Run All.

In [ ]:
import torch, time, sys, os
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}', f'({torch.cuda.get_device_name(0)})' if device=='cuda' else '')

In [ ]:
# Clone the repo (or upload the mxfp4-native package)
if not os.path.exists('memory-native-training'):
    !git clone -b research/mxfp4-native-no-state https://github.com/kharkilirov1/memory-native.git memory-native-training
sys.path.insert(0, 'memory-native-training/research/mxfp4-native-no-state/mxfp4-native')

from mx6.layer import MX6BlockLinear, MX6Linear
from mx6 import pack
print('mx6 imported OK')

In [ ]:
# Load Tiny Shakespeare
data_path = 'memory-native-training/research/mxfp4-native-no-state/mxfp4-native/data/tinyshakespeare.txt'
text = open(data_path, encoding='utf-8').read()
chars = sorted(set(text))
vocab = {c: i for i, c in enumerate(chars)}
V = len(chars)
ids = torch.tensor([vocab[c] for c in text], dtype=torch.long, device=device)
print(f'vocab: {V}, corpus: {len(ids)} chars')

In [ ]:
import torch.nn as nn

D = 128       # embedding dim (matches PHASE2 d=128)
SEQ = 128     # sequence length
BS = 32       # batch size
STEPS = 3000  # training steps (matches PHASE2)

class CharLM(nn.Module):
    def __init__(self, arm='block', D=D, V=V):
        super().__init__()
        self.embed = nn.Embedding(V, D)
        if arm == 'block':
            self.fc1 = MX6BlockLinear(D, D*4, lr=0.01, C_block=32, scale_headroom=1.5)
            self.fc2 = MX6BlockLinear(D*4, D, lr=0.01, C_block=32, scale_headroom=1.5)
        elif arm == 'sr':
            self.fc1 = MX6Linear(D, D*4, lr=0.01, integrator=0, scale_headroom=1.5)
            self.fc2 = MX6Linear(D*4, D, lr=0.01, integrator=0, scale_headroom=1.5)
        elif arm == '9state':
            self.fc1 = MX6Linear(D, D*4, lr=0.01, integrator=9, scale_headroom=1.5)
            self.fc2 = MX6Linear(D*4, D, lr=0.01, integrator=9, scale_headroom=1.5)
        elif arm == '3state':
            self.fc1 = MX6Linear(D, D*4, lr=0.01, integrator=3, scale_headroom=1.5)
            self.fc2 = MX6Linear(D*4, D, lr=0.01, integrator=3, scale_headroom=1.5)
        elif arm == 'adam':
            self.fc1 = MX6Linear(D, D*4, lr=0.01, integrator=0, scale_headroom=1.5)
            self.fc2 = MX6Linear(D*4, D, lr=0.01, integrator=0, scale_headroom=1.5)
        self.ln = nn.LayerNorm(D)
        self.head = nn.Linear(D, V, bias=False)
        self.arm = arm
        if arm == 'adam':
            self.opt = torch.optim.AdamW(
                list(self.embed.parameters()) + list(self.ln.parameters()) +
                list(self.head.parameters()) +
                [p for n,p in self.fc1.named_parameters() if 'codes' not in n and 'scale' not in n and 'acc' not in n and 'v_rms' not in n] +
                [p for n,p in self.fc2.named_parameters() if 'codes' not in n and 'scale' not in n and 'acc' not in n and 'v_rms' not in n],
                lr=3e-3)
        else:
            self.opt = torch.optim.SGD(
                list(self.embed.parameters()) + list(self.ln.parameters()) +
                list(self.head.parameters()), lr=0.01)

    def forward(self, x):
        h = self.embed(x)
        h = self.fc1(h)
        h = torch.relu(h)
        h = self.fc2(h)
        h = self.ln(h)
        return self.head(h)

def run(model, steps=STEPS, seed=0):
    torch.manual_seed(seed)
    model.to(device)
    model.train()
    for i in range(steps):
        idx = torch.randint(0, len(ids) - SEQ - 1, (BS,), device=device)
        x = ids[idx.unsqueeze(1) + torch.arange(SEQ, device=device)]
        y = ids[idx.unsqueeze(1) + torch.arange(1, SEQ+1, device=device)]
        logits = model(x)
        loss = nn.functional.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
        model.opt.zero_grad()
        loss.backward()
        model.opt.step()
    # eval
    model.eval()
    with torch.no_grad():
        idx = torch.randint(0, len(ids) - SEQ - 1, (BS,), device=device)
        x = ids[idx.unsqueeze(1) + torch.arange(SEQ, device=device)]
        y = ids[idx.unsqueeze(1) + torch.arange(1, SEQ+1, device=device)]
        logits = model(x)
        loss = nn.functional.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    return loss.item()

In [ ]:
# Run all arms, 3 seeds each
arms = ['sr', 'block', '3state', '9state', 'adam']
results = {}
for arm in arms:
    vals = []
    for seed in range(3):
        model = CharLM(arm=arm)
        t0 = time.time()
        val = run(model, steps=STEPS, seed=seed)
        dt = time.time() - t0
        vals.append(val)
        print(f'  {arm:8s} seed={seed}  val={val:.4f}  ({dt:.0f}s)')
    results[arm] = vals
    print(f'  {arm:8s} mean={sum(vals)/len(vals):.4f}')

In [ ]:
# Summary table
print(f'{'arm':<12} {'bits/param':>10} {'seed0':>8} {'seed1':>8} {'seed2':>8} {'mean':>8}')
print('-' * 58)
bpp = {'sr': 4.25, 'block': 4.50, '3state': 6.25, '9state': 8.25, 'adam': 100}
for arm in arms:
    v = results[arm]
    m = sum(v)/len(v)
    print(f'{arm:<12} {bpp[arm]:>10.2f} {v[0]:>8.4f} {v[1]:>8.4f} {v[2]:>8.4f} {m:>8.4f}')
print()
print('Lower is better.  Adam-QAT is the reference (fp32 master + AdamW).')
print('Block-acc target: beat 9-state at 4.50 bits/param vs 8.25.')

In [ ]:
# Bits-per-param measurement (actual, not idealised)
for arm in ['block', '3state', '9state']:
    model = CharLM(arm=arm)
    if arm == 'block':
        bpp = model.fc1.bits_per_param()
    else:
        # MX6Linear doesn't have bits_per_param, compute manually
        bpp = model.fc1.persistent_bytes() * 8 / (model.fc1.in_features * model.fc1.out_features)
    print(f'{arm:8s}  bits/param: {bpp:.3f}')